# **Notebook for Low VP Solids and Liquids Test set**

## **Processing National Bureau of Standards Technical Note Octoboer 1960** 
## *Vapor Pressure of Organic Compounds in the Range Below One Millimeter of Mercury*

**I manually digitized a scanned in version of this document. It contained 334 entries, though the age of the dataset makes it difficult to pull SMILES strings and find up-to-date IUPAC names and such. Some of the compounds are underspecified in terms of stereochemistry, so the dataset shrinks as it is processed for its usable data.**

**----------------------------------------------------------------------------------------------------------------------------------------**

### Loading in the 1960-70s Dataset that I manually digitized

In [ ]:
import pandas as pd
import numpy as np

# -----------------------------
# User input
# -----------------------------
excel_path = "./Raw Data/Vapor Pressures of Organic Compounds in the Range Below One Millimeter of Mercury.xlsx"
sheet_name = 0   # or name of the sheet

# -----------------------------
# Load raw Excel (no assumptions)
# -----------------------------
raw_df = pd.read_excel(
    excel_path,
    sheet_name=sheet_name,
    header=0,
    dtype=object,  # keep everything flexible initially
    engine = "openpyxl"
)

# Drop completely empty rows
raw_df = raw_df.dropna(how="all").reset_index(drop=True)

# -----------------------------
# Identify concentration columns
# -----------------------------
fixed_cols = ["Formula", "Name", "Phase", "Method", "Reference"]
concentration_cols = [
    c for c in raw_df.columns
    if c not in fixed_cols
]

# Convert concentration columns to numeric where possible
for col in concentration_cols:
    raw_df[col] = pd.to_numeric(raw_df[col], errors="coerce")

# -----------------------------
# Identify oddball / merged rows
# -----------------------------
# Criteria:
# 1. Only one non-null cell OR
# 2. No numeric data in concentration columns
is_oddball = (
    (raw_df.notna().sum(axis=1) <= 1) |
    (raw_df[concentration_cols].notna().sum(axis=1) == 0)
)

# -----------------------------
# Split data
# -----------------------------
data_df = raw_df.loc[~is_oddball].reset_index(drop=True)
oddball_df = raw_df.loc[is_oddball].reset_index(drop=True)

# -----------------------------
# Optional cleanup
# -----------------------------
# Strip whitespace from string columns
for col in fixed_cols:
    if col in data_df.columns:
        data_df[col] = data_df[col].astype(str).str.strip()

# -----------------------------
# Results
# -----------------------------
print(f"Data rows: {len(data_df)}")
print(f"Oddball rows: {len(oddball_df)}")


### Looking Up SMILES strings for Each Entry

#### NIH Smiles Query (Robust but time consuming)

In [ ]:
import requests
from rdkit import Chem
import pandas as pd
import urllib.parse
import os
from time import sleep

SAVE_PATH = "data_df_with_smiles_checkpoint.csv"
SAVE_EVERY = 10


# -----------------------------
# API Query Functions
# -----------------------------

def get_smiles_name(name):
    if pd.isna(name):
        return None

    name_enc = urllib.parse.quote(str(name))
    url = f"https://cactus.nci.nih.gov/chemical/structure/{name_enc}/smiles"

    try:
        r = requests.get(url, timeout=5)
        if r.status_code == 200:
            smi = r.text.strip()
            if smi:
                return smi
    except Exception:
        pass

    return None


def get_smiles_formula(formula):
    if pd.isna(formula):
        return None

    formula_enc = urllib.parse.quote(str(formula))
    url = f"https://cactus.nci.nih.gov/chemical/structure/{formula_enc}/smiles"

    try:
        r = requests.get(url, timeout=5)
        if r.status_code == 200:
            smi = r.text.strip()
            if smi:
                return smi
    except Exception:
        pass

    return None


# -----------------------------
# RDKit Canonicalization
# -----------------------------

def canonicalize_smiles(smiles):
    try:
        if smiles is None:
            return None

        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None

        return Chem.MolToSmiles(mol, canonical=True)

    except Exception:
        return None


# -----------------------------
# Row Resolver
# -----------------------------

def resolve_smiles(row):
    smi = get_smiles_name(row['Name'])

    if smi is None:
        smi = get_smiles_formula(row['Formula'])

    smi = canonicalize_smiles(smi)

    return smi


# -----------------------------
# Resume-safe Processing Loop
# -----------------------------

# Resume if checkpoint exists
if os.path.exists(SAVE_PATH):
    print("Loading checkpoint...")
    data_df = pd.read_csv(SAVE_PATH)
else:
    data_df['SMILES'] = None


processed = data_df['SMILES'].notna().sum()
print(f"Resuming at row: {processed}")


for idx in range(processed, len(data_df)):

    row = data_df.iloc[idx]

    smi = resolve_smiles(row)
    data_df.at[idx, 'SMILES'] = smi

    # Save every N rows
    if (idx + 1) % SAVE_EVERY == 0:
        data_df.to_csv(SAVE_PATH, index=False)
        print(f"Saved checkpoint at row {idx+1}")

    # Optional: be polite to NIH servers
    sleep(0.2)


# -----------------------------
# Final Cleanup
# -----------------------------

data_df = data_df[data_df['SMILES'].notna()].copy()
data_df = data_df.drop_duplicates(subset='SMILES', keep='first').reset_index(drop=True)

data_df.to_csv("1960s_NBoS_with_SMILES.csv", index=False)

print(f"Finished. Valid SMILES count: {len(data_df)}")


### **RUN FROM HERE IF MAKING MODIFICATIONS AFTER SMILES PULLING: Loading 1960s National Beureau of Standards Dataset with SMILES Strings Pulled from NIH**

In [ ]:
smiles_df = pd.read_csv("1960s_NBoS_with_SMILES.csv")
smiles_df = smiles_df[smiles_df["SMILES"].notna()]

display(smiles_df[smiles_df["Phase"] != "s"])

### Splitting into Liquid and solid Dataset

In [ ]:
# Identify pressure/temperature columns by checking column names are numeric
pressure_cols = []
for col in smiles_df.columns:
    try:
        float(col)
        pressure_cols.append(col)
    except:
        pass

# Split base data
liquid_df = smiles_df[smiles_df["Phase"] == "l"].copy()
solid_df  = smiles_df[smiles_df["Phase"] == "s"].copy()

all_cols = smiles_df.columns.tolist()

# Identify transition rows (Phase column contains melting temperature)
transition_mask = pd.to_numeric(smiles_df["Phase"], errors="coerce").notna()
transition_df = smiles_df[transition_mask].copy()
transition_df["Tm"] = pd.to_numeric(transition_df["Phase"])

solid_rows = []
liquid_rows = []

for _, row in transition_df.iterrows():

    melt_T = row["Tm"]

    # Important: keep ALL original columns
    solid_row = row.copy()
    liquid_row = row.copy()

    for col in pressure_cols:

        T = pd.to_numeric(row[col], errors="coerce")
        if pd.isna(T):
            continue

        # Liquid region
        if T >= melt_T:
            solid_row[col] = np.nan

        # Solid region
        else:
            liquid_row[col] = np.nan

    solid_row["Phase"] = "s"
    liquid_row["Phase"] = "l"

    solid_rows.append(solid_row)
    liquid_rows.append(liquid_row)

# Build transition DataFrames
transition_solid_df  = pd.DataFrame(solid_rows)
transition_liquid_df = pd.DataFrame(liquid_rows)

# ---- CRITICAL FIX ----
# Force same column structure so reference + pubchem_cid survive
transition_solid_df  = transition_solid_df.reindex(columns=all_cols)
transition_liquid_df = transition_liquid_df.reindex(columns=all_cols)

# Concatenate back
solid_df  = pd.concat([solid_df, transition_solid_df], ignore_index=True)
liquid_df = pd.concat([liquid_df, transition_liquid_df], ignore_index=True)


In [ ]:
# We can spot check some entries that we know underwent phase transition.
# It should be noted that the boiling point reported by this dataset is incorrect.
# We understand that the data is not perfect, but it is all we can find and work with.

display(solid_df[solid_df["Name"] == "1-butene"])
display(liquid_df[liquid_df["Name"] == "1-butene"])

solid_df.head()
print("Solid Entries:", len(solid_df))
print("Liquid Entries:", len(liquid_df))

### Fitting Antoine Coefficients to Possible datapoints.

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

# Antoine equation using Celsius
def antoine_eq(T, A, B, C):
    return A - (B / (T + C))


def fit_antoine_from_df(
    df,
    min_points=3,
    rmse_max=0.15,   # typical good fits ~0.03–0.08
):
    """
    Fits Antoine coefficients with RMSE quality filtering.

    RMSE is computed in log10(mmHg ~ torr) space.

    Returns dataframe of accepted fits only.
    """

    # Identify pressure columns
    pressure_cols = []
    for col in df.columns:
        try:
            float(col)
            pressure_cols.append(col)
        except:
            pass

    results = []

    for _, row in df.iterrows():

        T_vals = []
        P_vals = []

        for col in pressure_cols:

            T = pd.to_numeric(row[col], errors="coerce")
            if pd.isna(T):
                continue

            # microns Hg → mm Hg
            P_mmHg = float(col) / 1000.0

            if P_mmHg <= 0:
                continue

            T_vals.append(T)
            P_vals.append(P_mmHg)

        # Require minimum datapoints
        if len(T_vals) < min_points:
            continue

        T_vals = np.array(T_vals)
        P_vals = np.array(P_vals)

        logP = np.log10(P_vals)

        # Initial guess
        p0 = [8.0, 1500.0, 200.0]

        try:
            popt, _ = curve_fit(
                antoine_eq,
                T_vals,
                logP,
                p0=p0,
                maxfev=20000
            )

            A, B, C = popt

        except RuntimeError:
            continue

        # Predictions
        logP_pred = antoine_eq(T_vals, A, B, C)

        # RMSE in log space
        rmse = np.sqrt(np.mean((logP - logP_pred) ** 2))

        # Quality filter
        if rmse > rmse_max:
            continue

        Tmin = T_vals.min()
        Tmax = T_vals.max()

        results.append({
            "Formula": row.get("Formula"),
            "Name": row.get("Name"),
            "Phase": row.get("Phase"),

            "A": A,
            "B": B,
            "C": C,

            "Tmin_C": Tmin,
            "Tmax_C": Tmax,

            "RMSE_log10": rmse,
            "n_points": len(T_vals),

            "Method": row.get("Method"),
            "Reference": row.get("Reference"),

            "smiles": row.get("SMILES"),
            "data_source": "Hughes and Lias"
        })

    coef_df = pd.DataFrame(results)

    return coef_df


In [ ]:
solid_antoine_df = fit_antoine_from_df(solid_df)
solid_antoine_df.to_csv("./Useable Data/Hughes_Lias_Low_VP_Solids")
liquid_antoine_df = fit_antoine_from_df(liquid_df)
liquid_antoine_df.to_csv("./Useable Data/Hughes_Lias_Low_VP_Liquids")
print("Reduced # of entries in solid df:", len(solid_antoine_df))
print("Reduced # of entries in liquid df:", len(liquid_antoine_df))
display(solid_antoine_df.head())

# Processing the Explosives Dataset to find solid and liquid Antoine's Coefficients 

I've assessed that this dataset is not even worth using. Most of the entries are repative and there are only a couple dozen unique compounds.

In [ ]:
explosivo = pd.read_csv(
    "./Raw Data/explosives_paper_raw_data.csv",
    encoding="cp1252"
    )
display(explosivo.head(20))

solid_explosivo = explosivo[explosivo["Tmax [C]"] <= explosivo["mp [C]"]]
liquid_explosivo = explosivo[explosivo["Tmax [C]"] > explosivo["mp [C]"]]

print(len(explosivo))
print(len(liquid_explosivo))
print(len(solid_explosivo))

In [ ]:
solid_dupes = (
    solid_explosivo["abbr"]
    .value_counts()
    .loc[lambda x: x > 1]
)

print("Solid duplicate compounds:", len(solid_dupes))
display(solid_dupes)

liquid_dupes = (
    liquid_explosivo["abbr"]
    .value_counts()
    .loc[lambda x: x > 1]
)

print("Liquid duplicate compounds:", len(liquid_dupes))
display(liquid_dupes)